In [2]:
import pandas as pd

sp = pd.read_csv("../data/external/sponsors.csv")
print(sp.shape)
print(sp.columns.tolist())
print(sp.head(5).to_string())

(142701, 5)
['Organisation Name', 'Town/City', 'County', 'Type & Rating', 'Route']
                  Organisation Name    Town/City    County      Type & Rating           Route
0                               ALT      Glasgow       NaN  Worker (A rating)  Skilled Worker
1          Asian African Foods Ltd      London         NaN  Worker (A rating)  Skilled Worker
2                  BOLTWHIZ LIMITED  Dunfermline  Scotland  Worker (A rating)  Skilled Worker
3   Bossmans Retail Abergavenny Ltd  Abergavenny       NaN  Worker (A rating)  Skilled Worker
4        BRANOS OXFORD LTD T/A LILO       Oxford       NaN  Worker (A rating)  Skilled Worker


In [3]:
print(sp["Route"].value_counts())
print()
print(sp["Organisation Name"].nunique())

Route
Skilled Worker                                           122767
Global Business Mobility: Senior or Specialist Worker     10416
Tier 2 Ministers of Religion                               1937
Creative Worker                                            1581
Charity Worker                                             1527
International Sportsperson                                 1474
Religious Worker                                           1423
Global Business Mobility: Graduate Trainee                  616
Global Business Mobility: UK Expansion Worker               419
Government Authorised Exchange                              247
International Agreement                                     134
Scale-up                                                     92
Global Business Mobility: Service Supplier                   54
Seasonal Worker                                               6
Global Business Mobility: Secondment Worker                   6
Intra-company Routes              

In [4]:
skilled = sp[sp["Route"] == "Skilled Worker"].copy()
print(len(skilled))

meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
print(meta["company_name"].nunique())

direct = set(meta["company_name"]) & set(skilled["Organisation Name"])
print("直接匹配:", len(direct))
print(list(direct)[:20])

122767
658
直接匹配: 21
['Universities Superannuation Scheme Limited', 'UK Research and Innovation', 'Vitality Corporate Services Limited', 'Robert Half Limited', 'Gazelle Global Consulting Ltd', 'Royal Bank of Canada', 'Handle Recruitment', 'Department for Work and Pensions', 'Recycleye', 'Financial Ombudsman Service', 'Methods Business and Digital Technology', 'Ofcom', 'Morgan McKinley', 'N Consulting Ltd', 'British Cycling', 'Connected Places Catapult', 'Kroo Bank Ltd', 'National Highways', 'VIQU Limited', 'Contentsquare']


In [5]:
import re

SUFFIXES = r"\b(?:limited|ltd|llp|plc|inc|incorporated|corp|corporation|" \
           r"company|co|group|holdings|uk|gb|international|services|" \
           r"solutions|technologies|technology)\b"

def norm(name):
    if not isinstance(name, str):
        return ""
    s = name.lower()
    s = re.sub(r"\bt/a\b.*", "", s)        # 去掉 "trading as" 及其后内容
    s = s.replace("&", "and")
    s = re.sub(r"[^\w\s]", " ", s)          # 标点转空格
    s = re.sub(SUFFIXES, " ", s)            # 去掉公司后缀
    s = re.sub(r"\s+", " ", s).strip()      # 压缩空白
    return s

for t in ["Robert Half Limited", "BRANOS OXFORD LTD T/A LILO",
          "Deliveroo", "Anson Mccade", "Anson McCade",
          "IT Online Learning", "Turner & Townsend"]:
    print(f"{t:35s} → {norm(t)}")

Robert Half Limited                 → robert half
BRANOS OXFORD LTD T/A LILO          → branos oxford
Deliveroo                           → deliveroo
Anson Mccade                        → anson mccade
Anson McCade                        → anson mccade
IT Online Learning                  → it online learning
Turner & Townsend                   → turner and townsend


In [6]:
skilled["norm"] = skilled["Organisation Name"].apply(norm)
meta["norm"] = meta["company_name"].apply(norm)

sponsor_set = set(skilled["norm"]) - {""}
print("名单归一化后不重复:", len(sponsor_set))

companies = meta[["company_name", "norm"]].drop_duplicates("company_name")
companies["licensed"] = companies["norm"].isin(sponsor_set)

print("我的公司数:", len(companies))
print("匹配上:", companies["licensed"].sum())
print("匹配率:", round(companies["licensed"].mean() * 100, 1), "%")

名单归一化后不重复: 120465
我的公司数: 658
匹配上: 263
匹配率: 40.0 %


In [7]:
matched = companies[companies["licensed"]]["company_name"].tolist()
unmatched = companies[~companies["licensed"]]["company_name"].tolist()

print("=== 匹配上的前 30 ===")
for c in matched[:30]:
    print(" ", c)
print()
print("=== 没匹配上的前 30 ===")
for c in unmatched[:30]:
    print(" ", c)

=== 匹配上的前 30 ===
  Sphere Digital Recruitment
  Wise
  FDM Group
  ConvaTec
  Robert Half
  Robert Walters
  Adecco
  Ntt Data
  MONY Group
  Beyond
  Euro Car Parks
  Global Media Group
  Trace
  Exclusive Networks
  GoHenry
  AvePoint
  Funding Circle
  Sagacity
  Viber
  Crown Agents Bank
  Howden
  Apex Group
  Soho House
  Tabeo
  iProov
  Kraken
  Saffery
  Moniepoint
  Salary Finance
  Teya

=== 没匹配上的前 30 ===
  Harnham - Data & Analytics Recruitment
  G-Research
  Solirius Reply
  ITOL Recruit
  Marc Daniels
  Consula Group LTD
  Tenth Revolution Group
  Trace | Expert Accountancy & Finance Recruitment
  Lorien
  VIQU IT Recruitment
  LexisNexis
  Lendable
  OneForma
  Precise Placements
  Regal Brooke Limited
  Millennium Hotel and Resorts UK
  Hypercreate Ltd
  Millennium Hotels and Resorts
  Nomia
  Elevate Recruitment Limited
  UCL Partners
  Air Apps
  Winton
  Talan
  YO AI Labs
  Hometrack
  Zego
  Parker B Associates
  Micro IT Global
  Method Resourcing


In [8]:
AGENCY_PAT = r"\b(?:recruit\w*|recruiting|staffing|resourc\w*|talent|" \
             r"search|selection|consultan\w*|associates|partners|" \
             r"personnel|placements?|headhunt\w*)\b"

companies["is_agency"] = companies["company_name"].str.lower().str.contains(AGENCY_PAT, regex=True)

print(pd.crosstab(companies["is_agency"], companies["licensed"]))
print()
print("中介占比:", round(companies["is_agency"].mean() * 100, 1), "%")

licensed   False  True 
is_agency              
False        331    254
True          64      9

中介占比: 11.1 %


In [9]:
meta2 = meta.merge(companies[["company_name", "licensed", "is_agency"]],
                   on="company_name", how="left")
print("按岗位数：")
print(pd.crosstab(meta2["is_agency"], meta2["licensed"], normalize=True).round(3) * 100)

按岗位数：
licensed   False  True 
is_agency              
False       48.1   38.1
True        12.2    1.6


In [10]:
companies["norm_len"] = companies["norm"].str.split().str.len()
short = companies[companies["licensed"] & (companies["norm_len"] <= 1)]
print("单词名匹配上的:", len(short))
print(short["company_name"].tolist())

单词名匹配上的: 156
['Wise', 'FDM Group', 'ConvaTec', 'Adecco', 'MONY Group', 'Beyond', 'Trace', 'GoHenry', 'AvePoint', 'Sagacity', 'Viber', 'Howden', 'Apex Group', 'Tabeo', 'iProov', 'Kraken', 'Saffery', 'Moniepoint', 'Teya', 'Modulr', 'VIQU Limited', 'Man Group', 'NextEnergy Group', 'RSM', 'Liberis', 'Trustpilot', '9fin', 'Contentsquare', 'Tripadvisor', 'Cluttons', 'Synpulse', 'Ferrero', 'Man Group plc', 'PATRIZIA', 'ING', 'Markel', 'Autotrader', 'Ofcom', 'Nexperia', 'Canopius', 'Canopius Services', 'RSM UK', 'AECOM', 'NCC Group', 'THG', 'QA', 'Hays Technology', 'Addepar', 'iCapital', 'EcoOnline', 'Likewize', 'VM2R Services', 'Tumelo', 'Leonardo', 'Tekever', 'Multiverse', 'Hackajob Ltd', 'Google', 'Dunnhumby Ltd', 'VML', 'Cint', 'Damia Group Ltd', 'Wheely', 'Informed Solutions', 'LoopMe', 'Salt', 'RELX INC', 'Qogita', 'MarketCast', 'CoMind', 'marshmallow', 'dunnhumby', 'Ripple', 'Shift Technology', 'Carwow', 'Chainalysis', 'GoCardless', 'Wayve', 'Engitix', 'Fin', 'Systemiq', 'Preply', 'IBM'

In [11]:
for c in short["company_name"].head(8):
    n = norm(c)
    hits = skilled[skilled["norm"] == n]["Organisation Name"].unique()[:4]
    print(f"{c}  →  {n}")
    for h in hits:
        print("      ", h)
    print()


Wise  →  wise
       UK Wise Group Ltd

FDM Group  →  fdm
       FDM Group Limited

ConvaTec  →  convatec
       ConvaTec Ltd

Adecco  →  adecco
       Adecco UK Limited

MONY Group  →  mony
       MONY Group PLC

Beyond  →  beyond
       Beyond Holdings Limited

Trace  →  trace
       Trace Group Limited

GoHenry  →  gohenry
       GoHenry Limited



In [13]:
COMMON_WORDS = {
    "wise", "beyond", "trace", "salt", "fin", "dex", "betty", "prophet",
    "ki", "gamma", "sky", "snap", "google", "amazon", "citi", "swift",
    "apex", "howden", "markel", "qa", "rsm", "ing", "thg", "bp",
}

def confidence(row):
    if not row["licensed"]:
        return "unmatched"
    n = row["norm"]
    if len(n) <= 3:
        return "low"
    if " " not in n and n in COMMON_WORDS:
        return "low"
    if " " not in n and len(n) <= 6:
        return "medium"
    return "high"

companies["confidence"] = companies.apply(confidence, axis=1)
print(companies["confidence"].value_counts())

confidence
unmatched    395
high         183
medium        48
low           32
Name: count, dtype: int64


In [14]:
meta3 = meta.merge(
    companies[["company_name", "licensed", "is_agency", "confidence"]],
    on="company_name", how="left"
)

print("按岗位数：")
print(meta3["confidence"].value_counts())
print()
print((meta3["confidence"].value_counts(normalize=True) * 100).round(1))
print()
print("非中介 + high:", ((meta3["confidence"] == "high") & (~meta3["is_agency"])).sum())

按岗位数：
confidence
unmatched    806
high         365
medium        89
low           76
Name: count, dtype: int64

confidence
unmatched    60.3
high         27.3
medium        6.7
low           5.7
Name: proportion, dtype: float64

非中介 + high: 344


In [15]:
out = meta3[["id", "title", "company_name", "region", "licensed",
             "confidence", "is_agency"]]
out.to_csv("../data/processed/sponsor_match_20260812.csv", index=False)
print(out.shape)

(1336, 7)
